# 01 Run Benchmark

Run a sweep and generate benchmark figures + statistical summaries + paper claims.


In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)


Project root: /Users/adityadutta/Desktop/GitHub/ess-ope-diagnostics
Python: /Users/adityadutta/Desktop/GitHub/ess-ope-diagnostics/.venv/bin/python


In [2]:
import pandas as pd

from ess_ope.evaluation.sweep import SweepConfig, run_sweep
from ess_ope.evaluation.summary import (
    PaperClaimConfig,
    SummaryConfig,
    build_condition_summary,
    build_estimator_summary,
    build_paper_claim_summary,
    build_paper_claims_table,
)
from ess_ope.plotting.benchmark_figures import generate_benchmark_figures
from ess_ope.utils.config import load_yaml

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 220)


## Choose config
- `random_mdp_baseline.yaml`: fast
- `random_mdp_ultra.yaml`: maximum intensity
- `random_mdp_intensive.yaml`: heavy run


In [3]:
cfg_path = PROJECT_ROOT / "configs/sweeps/random_mdp_robust.yaml"
cfg = SweepConfig.from_dict(load_yaml(cfg_path))
cfg


SweepConfig(name='random_mdp_robust', results_root='results', gamma=1.0, temperature=1.0, seeds=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], alphas=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], betas=[0.0, 0.25, 0.5, 1.0], dataset_sizes=[100, 500], env_repeats=2, policy_repeats=2, dataset_repeats=8, num_workers=4, mp_chunksize=2, env={'num_states': 100, 'num_actions': 6, 'horizon': 20, 'branch_factor': 3, 'linear_feature_dim': 12, 'reward_noise_std': 0.0}, fqe_l2_reg=0.0001, mrdr_l2_reg=0.001)

In [4]:
# Optional smoke override (keep False for ultra runs)
USE_SMOKE_OVERRIDE = False
if USE_SMOKE_OVERRIDE:
    cfg.name = "benchmark_smoke"
    cfg.seeds = [0, 1, 2]
    cfg.alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
    cfg.betas = [0.0, 0.5, 1.0]
    cfg.dataset_sizes = [100, 500]
    cfg.env_repeats = 1
    cfg.policy_repeats = 1
    cfg.dataset_repeats = 3

cfg


SweepConfig(name='random_mdp_robust', results_root='results', gamma=1.0, temperature=1.0, seeds=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], alphas=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], betas=[0.0, 0.25, 0.5, 1.0], dataset_sizes=[100, 500], env_repeats=2, policy_repeats=2, dataset_repeats=8, num_workers=4, mp_chunksize=2, env={'num_states': 100, 'num_actions': 6, 'horizon': 20, 'branch_factor': 3, 'linear_feature_dim': 12, 'reward_noise_std': 0.0}, fqe_l2_reg=0.0001, mrdr_l2_reg=0.001)

In [5]:
df, run_dir, result_path = run_sweep(cfg)
fig_dir = run_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

benchmark_report = generate_benchmark_figures(df, output_dir=fig_dir, fixed_alpha=None, fixed_beta=0.0)
summary_cfg = SummaryConfig(bootstrap_samples=16000, bootstrap_max_points=200000, show_progress=True)
estimator_summary = build_estimator_summary(df, config=summary_cfg)
condition_summary = build_condition_summary(df, show_progress=True)
paper_claims = build_paper_claims_table(df, estimator_summary, benchmark_report, config=PaperClaimConfig())
paper_claim_summary = build_paper_claim_summary(paper_claims)

benchmark_report.to_csv(fig_dir / "benchmark_report.csv", index=False)
estimator_summary.to_csv(fig_dir / "estimator_summary.csv", index=False)
condition_summary.to_csv(fig_dir / "condition_summary.csv", index=False)
paper_claims.to_csv(fig_dir / "paper_claims.csv", index=False)
paper_claim_summary.to_csv(fig_dir / "paper_claim_summary.csv", index=False)

print("Run dir:", run_dir)
print("Results:", result_path)
print("Figure dir:", fig_dir)
print("Rows:", len(df))
paper_claims


Sweep:random_mdp_robust:   0%|          | 0/28160 [00:00<?, ?it/s]

Sweep:random_mdp_robust: 100%|██████████| 28160/28160 [20:03<00:00, 23.41it/s]


Run dir: results/20260225_175228_random_mdp_robust
Results: results/20260225_175228_random_mdp_robust/sweep_results.parquet
Figure dir: results/20260225_175228_random_mdp_robust/figures
Rows: 28160


,claim_id,estimator,statement,metric,value,ci_low,ci_high,threshold,verdict,confidence,aux_metric_ess_cv,aux_metric_alpha_fixed,aux_metric_ess_ratio,aux_metric_beta_fixed
0,C1_IS_ESS_INFORMATIVE,IS-PDIS,ESS should negatively correlate with IS error.,"spearman(ESS_norm, abs_error)",-0.544578,-0.552961,-0.536170,ci_high < 0 (strong: < -0.2),supported,high,NaN,NaN,NaN,NaN
1,C2_DM_WEAK_ESS_CORR,DM,ESS should be weak/non-diagnostic for non-IS e...,"spearman(ESS_norm, abs_error)",-0.075467,-0.087116,-0.063990,|corr| <= 0.15 (primary effect-size threshold)...,supported,high,NaN,NaN,NaN,NaN
2,C2_FQE_WEAK_ESS_CORR,FQE,ESS should be weak/non-diagnostic for non-IS e...,"spearman(ESS_norm, abs_error)",-0.003667,-0.015279,0.008001,|corr| <= 0.15 (primary effect-size threshold)...,supported,high,NaN,NaN,NaN,NaN
3,C3_DM_SAME_ESS_DIFF_ERROR,DM,"At fixed alpha (similar ESS), error can differ...",error_range_over_beta / mean_abs_error,1.019111,NaN,NaN,ESS_CV <= 0.12 and rel_error_range >= 0.4 (str...,supported,medium,0.038876,0.9,NaN,NaN
4,C3_FQE_SAME_ESS_DIFF_ERROR,FQE,"At fixed alpha (similar ESS), error can differ...",error_range_over_beta / mean_abs_error,0.890200,NaN,NaN,ESS_CV <= 0.12 and rel_error_range >= 0.4 (str...,supported,medium,0.038876,0.9,NaN,NaN
5,C3_MRDR_SAME_ESS_DIFF_ERROR,MRDR,"At fixed alpha (similar ESS), error can differ...",error_range_over_beta / mean_abs_error,1.629736,NaN,NaN,ESS_CV <= 0.12 and rel_error_range >= 0.4 (str...,supported,medium,0.038876,0.9,NaN,NaN
6,C4_FQE_ESS_SHIFT_ERROR_STABLE,FQE,"Across alpha at fixed beta, ESS may vary stron...",error_range_over_alpha / mean_abs_error,0.010360,NaN,NaN,ESS_ratio >= 3.0 and rel_error_range <= 0.35 (...,supported,medium,NaN,NaN,143.859983,0.0
7,C2_MRDR_WEAK_ESS_CORR,MRDR,ESS should be weak/non-diagnostic for non-IS e...,"spearman(ESS_norm, abs_error)",-0.462313,-0.471573,-0.453046,|corr| <= 0.15 (primary effect-size threshold)...,not_supported,high,NaN,NaN,NaN,NaN
8,C4_DM_ESS_SHIFT_ERROR_STABLE,DM,"Across alpha at fixed beta, ESS may vary stron...",error_range_over_alpha / mean_abs_error,0.500597,NaN,NaN,ESS_ratio >= 3.0 and rel_error_range <= 0.35 (...,not_supported,medium,NaN,NaN,143.859983,0.0
9,C4_MRDR_ESS_SHIFT_ERROR_STABLE,MRDR,"Across alpha at fixed beta, ESS may vary stron...",error_range_over_alpha / mean_abs_error,0.841554,NaN,NaN,ESS_ratio >= 3.0 and rel_error_range <= 0.35 (...,not_supported,medium,NaN,NaN,143.859983,0.0
